In [188]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import datasets, layers, metrics, optimizers, Sequential


In [197]:
(x, y), (x_test, y_test) = datasets.fashion_mnist.load_data()
print(x.shape,x_test.shape)


(60000, 28, 28) (10000, 28, 28)


In [198]:
def preprocess(x, y):
    x = tf.cast(x, dtype=tf.float32)/255
    y = tf.cast(y, dtype=tf.int32)
    return x, y

batch_size = 128

db = tf.data.Dataset.from_tensor_slices((x, y))
db = db.map(preprocess).shuffle(10000).batch(batch_size)

db_test = tf.data.Dataset.from_tensor_slices((x_test, y_test))
db_test = db_test.map(preprocess).batch(batch_size)
# print(next(iter(db))[0].shape)#(128, 28, 28)
print(x_test.shape,y_test.shape)
print(x.shape,y.shape)

(10000, 28, 28) (10000,)
(60000, 28, 28) (60000,)


In [191]:
model = Sequential([
    layers.Dense(256,activation=tf.nn.relu),# 784->256
    layers.Dense(128,activation=tf.nn.relu),
    layers.Dense(64,activation=tf.nn.relu),
    layers.Dense(32,activation=tf.nn.relu),
    layers.Dense(10),
])

In [196]:
#创建一个输入用来构建参数
model.build(input_shape=[None,28*28])
model.summary()

next(iter(db))[0].shape,next(iter(db_test))[0].shape

Model: "sequential_30"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_150 (Dense)           (None, 256)               200960    
                                                                 
 dense_151 (Dense)           (None, 128)               32896     
                                                                 
 dense_152 (Dense)           (None, 64)                8256      
                                                                 
 dense_153 (Dense)           (None, 32)                2080      
                                                                 
 dense_154 (Dense)           (None, 10)                330       
                                                                 
Total params: 244,522
Trainable params: 244,522
Non-trainable params: 0
_________________________________________________________________


(TensorShape([128, 28, 28]), TensorShape([128, 128, 28, 28]))

In [193]:
optimizer = optimizers.Adam(learning_rate=0.001)
for epoch in range(30):
    for step,(x,y) in enumerate(db):
        #x:[b,28,28]  y:[b]
        x = tf.reshape(x,[-1,28*28])
        with tf.GradientTape() as tape:
            #x:[b,784]==>[b,10]
            logits = model(x)
            y_onehot = tf.one_hot(y,depth=10)
            loss_mse = tf.reduce_mean(tf.losses.MSE(y_onehot,logits))
            loss_ce = tf.losses.categorical_crossentropy(y_onehot,logits,from_logits=True)
            loss_ce = tf.reduce_mean(loss_ce)
        #自动求导！！！！
        grads = tape.gradient(loss_ce,model.trainable_variables)
        optimizer.apply_gradients(zip(grads,model.trainable_variables))
        if step%100 == 0 :
            print(epoch,step,'loss:',float(loss_mse),float(loss_ce))
        
    total_correct = 0
    total_num = 0 
    for x,y in db_test:
        x = tf.reshape(x,[-1,28*28]) 
        logits = model(x)
        #logits =>prob
        prob = tf.nn.softmax(logits,axis=1)
        pred = tf.argmax(prob,axis=1)#[b,10]=>[b]
        pred = tf.cast(pred,tf.int32)
        # print(y.shape,pred.shape)
        correct = tf.equal(pred,y)#[b],true,false
        correct = tf.reduce_sum(tf.cast(correct,tf.int32))
        # print(correct)
        total_correct+=int(correct)
        total_num+=x.shape[0]
    acc = total_correct/total_num
    print('*'*20)
    print(epoch,step,acc)
    print('*'*20)

0 0 loss: 0.15228337049484253 2.345454692840576
0 100 loss: 18.986194610595703 0.6241833567619324
0 200 loss: 17.060592651367188 0.5258536338806152
0 300 loss: 13.242576599121094 0.4526822865009308
0 400 loss: 20.847719192504883 0.4771037697792053


InvalidArgumentError: Incompatible shapes: [16384] vs. [128,128] [Op:Equal]